# Extrapolation Test: Train-Short / Test-Long

## Hypothesis

RoTHP's trigonometric kernel oscillates with period 2π in normalized time.  
HoTHP's hyperbolic kernel decays monotonically.

**If this matters**, it should be visible when the model is forced to compute
attention at temporal lags **it has never seen during training**.

## Why the previous experiment was too weak

With per-sequence normalization, the maximum normalized lag in any sequence is exactly
`n_events − 1`, regardless of raw timestamps or injected gaps.
When both train and test sequences had up to 200 events, the max lag was ≤ 199 in both
splits — no genuine temporal extrapolation.

## New design (pre-registered)

| Split | max_events | max normalized lag | Status |
|---|---|---|---|
| Train / Val / Test-short | 50 | ≤ 49 | in-distribution |
| Test-extrap | 500 | ≤ 499 | **OOD for positions > 50** |

No artificial gap injection. The sequence **length difference** is the extrapolation.

## Two pre-specified evaluation metrics

1. **overall_extrap_nll**: NLL averaged over *all* events in the long test sequence.
2. **post_train_nll**: NLL averaged only over events at positions ≥ `TRAIN_MAX_EVENTS`
   in the long sequence. These events attend back to lags > 49 — genuinely OOD for
   both models. This metric is defined *before* running any experiment.

Both metrics are pre-specified. Neither is chosen post-hoc based on results.

In [ ]:
import os, sys, json, math, random, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

BASE_SEED = 42
TRAIN_MAX_EVENTS = 50   # training sequence length cap; also the OOD cutoff for post_train_nll
EXTRAP_MAX_EVENTS = 500 # test-extrap sequence length cap

def set_global_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

def make_run_seed(*parts, base_seed=BASE_SEED):
    key = '::'.join(map(str, parts))
    return (base_seed + int(hashlib.sha256(key.encode()).hexdigest()[:8], 16)) % (2**31)

set_global_seed(BASE_SEED)
sns.set_theme(style='whitegrid')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if not os.path.exists('ufc-easytpp'):
    !git clone https://github.com/hugoramos/ufc-easytpp.git
if 'ufc-easytpp' not in sys.path:
    sys.path.insert(0, os.path.abspath('ufc-easytpp'))

import easy_tpp.model.torch_model.torch_baselayer as baselayer

def attention_fixed(query, key, value, mask=None, dropout=None):
    d_k = query.size(-1)
    scores = torch.matmul(query, key.transpose(-2, -1)) / (d_k ** 0.5)
    if mask is not None:
        if mask.dim() == 3: mask = mask.unsqueeze(1)
        scores = scores.masked_fill(mask > 0, -1e4)
    p_attn = torch.softmax(scores, dim=-1)
    if dropout is not None: p_attn = dropout(p_attn)
    return torch.matmul(p_attn, value), p_attn

baselayer.attention = attention_fixed
import easy_tpp.model.torch_model.torch_rothp as rothp_module
rothp_module.attention = attention_fixed

from easy_tpp.config_factory.model_config import ModelConfig
from easy_tpp.model.torch_model.torch_rothp import RoTHP
from easy_tpp.model.torch_model.torch_hothp import HoTHP

print(f'Device: {device}')
print(f'Train max events: {TRAIN_MAX_EVENTS}  →  max normalized lag ≈ {TRAIN_MAX_EVENTS - 1}')
print(f'Extrap max events: {EXTRAP_MAX_EVENTS}  →  max normalized lag ≈ {EXTRAP_MAX_EVENTS - 1}')

## Data Generation

Same Hawkes process parameters as before.  
The only difference: training sequences are capped at 50 events, extrapolation sequences at 500.

After per-sequence normalization (mean gap = 1.0):
- Train max lag ≈ 49
- Extrap max lag ≈ 499

Events at positions 50–499 in extrap sequences attend back to lags > 49 — never seen during training.

In [ ]:
NUM_TYPES = 4

mu    = np.array([0.12, 0.10, 0.09, 0.08])
alpha = np.array([
    [0.30, 0.08, 0.05, 0.03],
    [0.06, 0.28, 0.07, 0.04],
    [0.04, 0.06, 0.26, 0.06],
    [0.03, 0.04, 0.05, 0.24],
])
beta_fast, beta_slow = 2.5, 0.15
w_fast,    w_slow    = 0.6, 0.4


def generate_hawkes(rng, num_types, horizon, mu, alpha, beta_fast, beta_slow,
                    w_fast, w_slow, min_events=20, max_events=50):
    while True:
        events = []
        t = 0.0
        while t < horizon and len(events) < max_events:
            intensity = mu.copy()
            for t_i, k_i in events:
                dt = t - t_i
                kernel_val = w_fast * np.exp(-beta_fast * dt) + w_slow * np.exp(-beta_slow * dt)
                intensity += alpha[:, k_i] * kernel_val
            lam_bar = float(np.sum(intensity))
            if lam_bar <= 1e-9:
                break
            t += rng.exponential(1.0 / lam_bar)
            if t >= horizon:
                break
            candidate = mu.copy()
            for t_i, k_i in events:
                dt = t - t_i
                kernel_val = w_fast * np.exp(-beta_fast * dt) + w_slow * np.exp(-beta_slow * dt)
                candidate += alpha[:, k_i] * kernel_val
            lam_sum = float(np.sum(candidate))
            if rng.uniform() <= lam_sum / lam_bar:
                probs = candidate / lam_sum
                events.append((t, int(rng.choice(num_types, p=probs))))
        if len(events) >= min_events:
            return events[:max_events]


rng = np.random.default_rng(BASE_SEED)

# Short sequences: train / val / test-short
train_seqs = [
    generate_hawkes(rng, NUM_TYPES, horizon=15.0,
                    mu=mu, alpha=alpha, beta_fast=beta_fast, beta_slow=beta_slow,
                    w_fast=w_fast, w_slow=w_slow, min_events=20, max_events=TRAIN_MAX_EVENTS)
    for _ in tqdm(range(500), desc='Train')
]
val_seqs = [
    generate_hawkes(rng, NUM_TYPES, horizon=15.0,
                    mu=mu, alpha=alpha, beta_fast=beta_fast, beta_slow=beta_slow,
                    w_fast=w_fast, w_slow=w_slow, min_events=20, max_events=TRAIN_MAX_EVENTS)
    for _ in tqdm(range(100), desc='Val')
]
test_short_seqs = [
    generate_hawkes(rng, NUM_TYPES, horizon=15.0,
                    mu=mu, alpha=alpha, beta_fast=beta_fast, beta_slow=beta_slow,
                    w_fast=w_fast, w_slow=w_slow, min_events=20, max_events=TRAIN_MAX_EVENTS)
    for _ in tqdm(range(100), desc='Test-short')
]

# Long sequences: test-extrap
# Longer horizon (150) + higher max_events to get sequences of 100-500 events
test_extrap_seqs = [
    generate_hawkes(rng, NUM_TYPES, horizon=150.0,
                    mu=mu, alpha=alpha, beta_fast=beta_fast, beta_slow=beta_slow,
                    w_fast=w_fast, w_slow=w_slow, min_events=100, max_events=EXTRAP_MAX_EVENTS)
    for _ in tqdm(range(100), desc='Test-extrap (long)')
]

train_lengths  = [len(s) for s in train_seqs]
extrap_lengths = [len(s) for s in test_extrap_seqs]
print(f'Train sequence lengths:  mean={np.mean(train_lengths):.0f}, '
      f'min={min(train_lengths)}, max={max(train_lengths)}')
print(f'Extrap sequence lengths: mean={np.mean(extrap_lengths):.0f}, '
      f'min={min(extrap_lengths)}, max={max(extrap_lengths)}')
print(f'\nMax normalized lag in train:  ≈ {max(train_lengths) - 1}')
print(f'Max normalized lag in extrap: ≈ {max(extrap_lengths) - 1}')
print(f'\nExtrapolation zone: lags {TRAIN_MAX_EVENTS - 1}–{max(extrap_lengths) - 1} '
      f'(never seen during training)')

In [ ]:
def convert_to_tensors(seqs, num_types):
    """Per-sequence normalization: mean gap = 1.0.
    After this, times[-1] - times[0] = n_events - 1 exactly.
    """
    converted = []
    for seq in seqs:
        seq = sorted(seq, key=lambda x: x[0])
        times = torch.tensor([t for t, _ in seq], dtype=torch.float32)
        types = torch.tensor([k for _, k in seq], dtype=torch.long)
        deltas = torch.zeros_like(times)
        deltas[1:] = times[1:] - times[:-1]
        mean_gap = deltas[1:].mean().clamp(min=1e-6)
        times = (times - times[0]) / mean_gap
        deltas = deltas / mean_gap
        converted.append({'time_seqs': times, 'time_delta_seqs': deltas, 'type_seqs': types})
    return converted


def collate_fn(batch_list, pad_id):
    batch_size = len(batch_list)
    max_len = max(len(x['time_seqs']) for x in batch_list)
    pad_time  = torch.zeros(batch_size, max_len, dtype=torch.float32)
    pad_delta = torch.zeros(batch_size, max_len, dtype=torch.float32)
    pad_type  = torch.full((batch_size, max_len), pad_id, dtype=torch.long)
    non_pad_mask   = torch.zeros(batch_size, max_len, dtype=torch.float32)
    attention_mask = torch.ones(batch_size, max_len, max_len, dtype=torch.bool)
    causal = torch.triu(torch.ones(max_len, max_len, dtype=torch.bool), diagonal=1)
    for i, item in enumerate(batch_list):
        l = len(item['time_seqs'])
        pad_time[i, :l]  = item['time_seqs']
        pad_delta[i, :l] = item['time_delta_seqs']
        pad_type[i, :l]  = item['type_seqs']
        non_pad_mask[i, :l] = 1.0
        m = causal.clone()
        m[:, l:] = True
        m[l:, :] = True
        attention_mask[i] = m
    return (pad_time, pad_delta, pad_type, non_pad_mask, attention_mask)


train_data       = convert_to_tensors(train_seqs,       NUM_TYPES)
val_data         = convert_to_tensors(val_seqs,         NUM_TYPES)
test_short_data  = convert_to_tensors(test_short_seqs,  NUM_TYPES)
test_extrap_data = convert_to_tensors(test_extrap_seqs, NUM_TYPES)

pad_id  = NUM_TYPES
collate = lambda x: collate_fn(x, pad_id)

gen = torch.Generator()
gen.manual_seed(make_run_seed('length', 'loader'))
train_loader       = DataLoader(train_data,       batch_size=64,  shuffle=True,  collate_fn=collate, generator=gen)
val_loader         = DataLoader(val_data,         batch_size=64,  shuffle=False, collate_fn=collate)
test_short_loader  = DataLoader(test_short_data,  batch_size=32,  shuffle=False, collate_fn=collate)
test_extrap_loader = DataLoader(test_extrap_data, batch_size=16,  shuffle=False, collate_fn=collate)

# Verify lag ranges
def max_lag(data):
    return max(float(item['time_seqs'][-1] - item['time_seqs'][0]) for item in data)

print(f'Max normalized lag — Train: {max_lag(train_data):.1f}')
print(f'Max normalized lag — Test-short: {max_lag(test_short_data):.1f}')
print(f'Max normalized lag — Test-extrap: {max_lag(test_extrap_data):.1f}')
print(f'\nFraction of extrap events in OOD zone (position >= {TRAIN_MAX_EVENTS}):')
total_events = sum(len(item['time_seqs']) for item in test_extrap_data)
ood_events   = sum(max(0, len(item['time_seqs']) - TRAIN_MAX_EVENTS) for item in test_extrap_data)
print(f'  {ood_events}/{total_events} = {100*ood_events/total_events:.1f}%')

## Training

Key changes vs previous notebook:
- `ReduceLROnPlateau` scheduler for both models — reduces LR by 0.5 when val NLL plateaus
  for 10 epochs. This helps HoTHP escape the flat initial phase without changing the
  architecture or loss function.
- `post_train_nll`: NLL computed only for positions ≥ `TRAIN_MAX_EVENTS` in extrap sequences.
  This metric is pre-specified and directly tests the OOD lag regime.

In [ ]:
config = ModelConfig(**{
    'hidden_size': 32,
    'num_layers': 2,
    'num_heads': 2,
    'dropout_rate': 0.1,
    'num_event_types': NUM_TYPES,
    'num_event_types_pad': NUM_TYPES + 1,
    'event_pad_index': pad_id,
    'time_emb_size': 32,
    'use_ln': True,
    'gpu': 0 if torch.cuda.is_available() else -1,
    'model_id': 'LengthExtrap',
    'thinning': {
        'num_sample': 1, 'num_exp': 500, 'over_sample_rate': 5.0,
        'patience_counter': 5, 'num_samples_boundary': 5,
        'dtime_max': 5.0, 'num_step_gen': 1,
    },
    'loss_integral_num_sample_per_step': 20,
    'use_mc_samples': False,
})

USE_AMP = device.type == 'cuda'
import contextlib
if USE_AMP:
    try:
        _scaler_cls = torch.amp.GradScaler
        _autocast_fn = lambda: torch.amp.autocast(device_type='cuda')
    except AttributeError:
        _scaler_cls = torch.cuda.amp.GradScaler
        _autocast_fn = lambda: torch.cuda.amp.autocast()
else:
    _scaler_cls = None
    _autocast_fn = contextlib.nullcontext

print(f'AMP enabled: {USE_AMP}')


def evaluate_nll(model, loader):
    model.eval()
    total_loss, total_events = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            batch = [t.to(device) for t in batch]
            with _autocast_fn():
                loss, num = model.loglike_loss(batch)
            total_loss += loss.item()
            total_events += num
    return total_loss / (total_events + 1e-9)


def evaluate_post_nll(model, loader, cutoff=TRAIN_MAX_EVENTS):
    """NLL only for events at positions >= cutoff (the OOD lag zone).

    The model still sees the full sequence for attention context; we only
    exclude the first `cutoff` positions from the loss computation by
    zeroing their non_pad_mask entries.
    """
    model.eval()
    total_loss, total_events = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            pad_time, pad_delta, pad_type, non_pad_mask, attention_mask = \
                [t.to(device) for t in batch]
            # Zero out positions 0..cutoff-1 so they don't contribute to the loss
            post_mask = non_pad_mask.clone()
            post_mask[:, :cutoff] = 0.0
            # Skip batch if no OOD events (all sequences shorter than cutoff)
            if post_mask.sum() == 0:
                continue
            with _autocast_fn():
                loss, num = model.loglike_loss(
                    [pad_time, pad_delta, pad_type, post_mask, attention_mask])
            total_loss += loss.item()
            total_events += num
    return total_loss / (total_events + 1e-9)


def compute_spectral_weights(model, n_head, d_k):
    half = d_k // 2
    Wq = model.stack_layers[0].self_attn.linears[0].weight.detach().cpu()
    energies = []
    for j in range(half):
        e = 0.0
        for h in range(n_head):
            dim0 = h * d_k + 2 * j
            dim1 = h * d_k + 2 * j + 1
            e += Wq[dim0].pow(2).sum().item() + Wq[dim1].pow(2).sum().item()
        energies.append(e / n_head)
    total = sum(energies) + 1e-12
    return energies[0] / total, energies[-1] / total, energies[0] / (energies[-1] + 1e-12)


def train_model(model, name, train_loader, val_loader, epochs=600, patience=30,
                grad_clip=1.0, lr=1e-3, test_short_loader=None,
                test_extrap_loader=None, log_test_every=5):
    print(f'\nTraining {name} (lr={lr})...')
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='min', factor=0.5, patience=10, min_lr=1e-5)
    scaler = _scaler_cls(enabled=True) if USE_AMP else None

    best_val = float('inf')
    best_state = None
    no_improve = 0
    history = []

    for ep in range(epochs):
        model.train()
        for batch in tqdm(train_loader, desc=f'{name} Ep {ep+1}', leave=False):
            batch = [t.to(device) for t in batch]
            opt.zero_grad()
            with _autocast_fn():
                loss, num = model.loglike_loss(batch)
                nll = loss / (num + 1e-9)
            if not torch.isnan(nll):
                if scaler is not None:
                    scaler.scale(nll).backward()
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                    scaler.step(opt)
                    scaler.update()
                else:
                    nll.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                    opt.step()

        val_nll = evaluate_nll(model, val_loader)
        scheduler.step(val_nll)
        current_lr = opt.param_groups[0]['lr']

        rec = {'ep': ep + 1, 'val_nll': val_nll, 'lr': current_lr}
        if (test_short_loader is not None and test_extrap_loader is not None
                and (ep + 1) % log_test_every == 0):
            rec['test_short_nll']      = evaluate_nll(model, test_short_loader)
            rec['test_extrap_nll']     = evaluate_nll(model, test_extrap_loader)
            rec['test_post_train_nll'] = evaluate_post_nll(model, test_extrap_loader)
        history.append(rec)

        if val_nll < best_val - 1e-4:
            best_val = val_nll
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if (ep + 1) % 25 == 0:
            print(f'  Ep {ep+1}: Val={val_nll:.4f} (best={best_val:.4f}) lr={current_lr:.2e}')

        if no_improve >= patience:
            print(f'  Early stopping at epoch {ep+1}')
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return best_val, history


# ── Multi-seed training ──────────────────────────────────────────────────────
N_SEEDS = 10
EPOCHS  = 600
seeds   = [BASE_SEED + i * 100 for i in range(N_SEEDS)]
D_K     = config.hidden_size // config.num_heads

results_per_seed = []

for run, seed in enumerate(seeds):
    print(f'\n{"="*60}')
    print(f'Seed {run+1}/{N_SEEDS} (seed={seed})')
    print('='*60)

    set_global_seed(make_run_seed('length', 'RoTHP', base_seed=seed))
    rothp = RoTHP(config).to(device)
    rothp_val, rothp_history = train_model(
        rothp, 'RoTHP', train_loader, val_loader,
        epochs=EPOCHS, lr=1e-3,
        test_short_loader=test_short_loader,
        test_extrap_loader=test_extrap_loader,
        log_test_every=5,
    )
    fast_frac, slow_frac, spec_ratio = compute_spectral_weights(rothp, config.num_heads, D_K)
    print(f'  RoTHP spectral: fast={fast_frac:.3f}, slow={slow_frac:.3f}, ratio={spec_ratio:.2f}')

    set_global_seed(make_run_seed('length', 'HoTHP', base_seed=seed))
    hothp = HoTHP(config).to(device)
    hothp_val, hothp_history = train_model(
        hothp, 'HoTHP', train_loader, val_loader,
        epochs=EPOCHS, lr=5e-4,
        test_short_loader=test_short_loader,
        test_extrap_loader=test_extrap_loader,
        log_test_every=5,
    )

    rothp_short     = evaluate_nll(model=rothp, loader=test_short_loader)
    rothp_extrap    = evaluate_nll(model=rothp, loader=test_extrap_loader)
    rothp_post      = evaluate_post_nll(model=rothp, loader=test_extrap_loader)
    hothp_short     = evaluate_nll(model=hothp, loader=test_short_loader)
    hothp_extrap    = evaluate_nll(model=hothp, loader=test_extrap_loader)
    hothp_post      = evaluate_post_nll(model=hothp, loader=test_extrap_loader)

    results_per_seed.append({
        'seed': seed,
        'rothp_short':  rothp_short,  'rothp_extrap': rothp_extrap,  'rothp_post': rothp_post,
        'hothp_short':  hothp_short,  'hothp_extrap': hothp_extrap,  'hothp_post': hothp_post,
        'rothp_deg':    rothp_extrap  - rothp_short,
        'hothp_deg':    hothp_extrap  - hothp_short,
        'rothp_post_deg': rothp_post  - rothp_short,
        'hothp_post_deg': hothp_post  - hothp_short,
        'rothp_history': rothp_history,
        'hothp_history': hothp_history,
        'rothp_fast_frac': fast_frac,
        'rothp_slow_frac': slow_frac,
        'rothp_spec_ratio': spec_ratio,
    })
    r = results_per_seed[-1]
    print(f'  RoTHP: short={rothp_short:.4f}, extrap={rothp_extrap:.4f}, '
          f'post={rothp_post:.4f}  (deg={r["rothp_deg"]:+.4f}, post_deg={r["rothp_post_deg"]:+.4f})')
    print(f'  HoTHP: short={hothp_short:.4f}, extrap={hothp_extrap:.4f}, '
          f'post={hothp_post:.4f}  (deg={r["hothp_deg"]:+.4f}, post_deg={r["hothp_post_deg"]:+.4f})')

print(f'\nBest Val NLL (last seed) -> RoTHP: {rothp_val:.4f}  |  HoTHP: {hothp_val:.4f}')

In [ ]:
from scipy import stats

# ── Aggregate ────────────────────────────────────────────────────────────────
def arr(key): return np.array([r[key] for r in results_per_seed])

rothp_short_arr    = arr('rothp_short')
rothp_extrap_arr   = arr('rothp_extrap')
rothp_post_arr     = arr('rothp_post')
hothp_short_arr    = arr('hothp_short')
hothp_extrap_arr   = arr('hothp_extrap')
hothp_post_arr     = arr('hothp_post')
rothp_deg_arr      = arr('rothp_deg')
hothp_deg_arr      = arr('hothp_deg')
rothp_post_deg_arr = arr('rothp_post_deg')
hothp_post_deg_arr = arr('hothp_post_deg')

def ms(a): return a.mean(), a.std(ddof=1) if len(a) > 1 else 0.0

print(f'Results over {N_SEEDS} seeds')
print(f'{"":30} {"RoTHP":>14} {"HoTHP":>14}')
print(f'{"Short NLL (in-dist)":30} {ms(rothp_short_arr)[0]:.4f}±{ms(rothp_short_arr)[1]:.4f} '
      f'{ms(hothp_short_arr)[0]:.4f}±{ms(hothp_short_arr)[1]:.4f}')
print(f'{"Extrap NLL (all events)":30} {ms(rothp_extrap_arr)[0]:.4f}±{ms(rothp_extrap_arr)[1]:.4f} '
      f'{ms(hothp_extrap_arr)[0]:.4f}±{ms(hothp_extrap_arr)[1]:.4f}')
print(f'{"Post-train NLL (pos>=50)":30} {ms(rothp_post_arr)[0]:.4f}±{ms(rothp_post_arr)[1]:.4f} '
      f'{ms(hothp_post_arr)[0]:.4f}±{ms(hothp_post_arr)[1]:.4f}')
print(f'{"Degradation (extrap-short)":30} {ms(rothp_deg_arr)[0]:+.4f}±{ms(rothp_deg_arr)[1]:.4f} '
      f'{ms(hothp_deg_arr)[0]:+.4f}±{ms(hothp_deg_arr)[1]:.4f}')
print(f'{"Post-train degradation":30} {ms(rothp_post_deg_arr)[0]:+.4f}±{ms(rothp_post_deg_arr)[1]:.4f} '
      f'{ms(hothp_post_deg_arr)[0]:+.4f}±{ms(hothp_post_deg_arr)[1]:.4f}')

print('\n--- Spectral weight diagnostic ---')
print(f'{"Seed":>6}  {"spec_ratio":>10}  {"rothp_post_deg":>14}  {"hothp_post_deg":>14}')
for r in results_per_seed:
    print(f'{r["seed"]:>6}  {r["rothp_spec_ratio"]:>10.2f}  '
          f'{r["rothp_post_deg"]:>+14.4f}  {r["hothp_post_deg"]:>+14.4f}')

# ── Statistical tests ────────────────────────────────────────────────────────
print('\n--- Statistical tests (H1: RoTHP post_deg > HoTHP post_deg) ---')
for label, r_arr, h_arr in [
    ('All-events degradation', rothp_deg_arr, hothp_deg_arr),
    ('Post-train degradation', rothp_post_deg_arr, hothp_post_deg_arr),
]:
    t_stat, p_two = stats.ttest_rel(r_arr, h_arr)
    p_one = p_two / 2 if t_stat > 0 else 1.0 - p_two / 2
    try:
        _, p_w = stats.wilcoxon(r_arr, h_arr, alternative='greater')
        w_str = f'Wilcoxon p={p_w:.4f}'
    except Exception:
        w_str = 'Wilcoxon N/A'
    sig = '✓ significant' if p_one < 0.05 else '✗ not significant'
    print(f'  {label}: t={t_stat:.3f}, one-sided p={p_one:.4f}  {w_str}  [{sig}]')

# ── Figure 1: bar chart (3 splits) ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
bar_width = 0.35
x = np.arange(3)
r_means = [ms(rothp_short_arr)[0], ms(rothp_extrap_arr)[0], ms(rothp_post_arr)[0]]
r_stds  = [ms(rothp_short_arr)[1], ms(rothp_extrap_arr)[1], ms(rothp_post_arr)[1]]
h_means = [ms(hothp_short_arr)[0], ms(hothp_extrap_arr)[0], ms(hothp_post_arr)[0]]
h_stds  = [ms(hothp_short_arr)[1], ms(hothp_extrap_arr)[1], ms(hothp_post_arr)[1]]
bars1 = ax.bar(x - bar_width/2, r_means, bar_width, yerr=r_stds,
               label='RoTHP', color='#4C72B0', capsize=4)
bars2 = ax.bar(x + bar_width/2, h_means, bar_width, yerr=h_stds,
               label='HoTHP', color='#C44E52', capsize=4)
ax.set_xticks(x)
ax.set_xticklabels([
    'Short\n(in-dist, lag≤49)',
    'Extrap all events\n(lag≤499)',
    f'Extrap pos≥{TRAIN_MAX_EVENTS}\n(lag OOD zone)'
])
ax.set_ylabel('Test NLL (lower is better)')
ax.set_title(f'Train-short / Test-long extrapolation ({N_SEEDS} seeds)\n'
             f'Train: max {TRAIN_MAX_EVENTS} events (lag≤{TRAIN_MAX_EVENTS-1})  |  '
             f'Test-extrap: max {EXTRAP_MAX_EVENTS} events (lag≤{EXTRAP_MAX_EVENTS-1})')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('Extrapolation_Length_Test.png', dpi=300, bbox_inches='tight')
plt.show()

# ── Figure 2: per-seed post-train degradation ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x_pos = np.arange(N_SEEDS)
seed_labels = [str(r['seed']) for r in results_per_seed]

axes[0].plot(x_pos, rothp_post_deg_arr, 'o-', color='#4C72B0', lw=1.5, ms=7, label='RoTHP')
axes[0].plot(x_pos, hothp_post_deg_arr, 's--', color='#C44E52', lw=1.5, ms=7, label='HoTHP')
axes[0].axhline(0, color='gray', linestyle=':', alpha=0.6)
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(seed_labels, rotation=45, ha='right', fontsize=9)
axes[0].set_xlabel('Seed')
axes[0].set_ylabel(f'Post-train degradation (pos≥{TRAIN_MAX_EVENTS} NLL − short NLL)')
axes[0].set_title('Per-seed post-train degradation\n(OOD lag zone only)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

spec_ratios = arr('rothp_spec_ratio')
sc = axes[1].scatter(spec_ratios, rothp_post_deg_arr, c=rothp_post_deg_arr,
                     cmap='RdYlGn_r', s=80, zorder=3, edgecolors='k', lw=0.5)
plt.colorbar(sc, ax=axes[1], label='RoTHP post-train degradation')
axes[1].set_xlabel('Spectral ratio (fast / slow Q energy)')
axes[1].set_ylabel('RoTHP post-train degradation')
axes[1].set_title('Does Q frequency alignment predict\nextrapolation failure?')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Post-train NLL: genuine OOD lag evaluation', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('Length_Extrap_per_seed.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ── Learned attention profiles on test-extrap sequences ─────────────────────
def extract_attention_vs_lag(model, loader, model_type='rothp', max_samples=50):
    model.eval()
    lags, weights = [], []
    sample_count = 0
    with torch.no_grad():
        for batch in loader:
            if sample_count >= max_samples:
                break
            pad_time, pad_delta, pad_type, mask, attn_mask = [t.to(device) for t in batch]
            enc_output = model.layer_type_emb(pad_type)
            layer = model.stack_layers[0]

            if model_type == 'rothp':
                cos, sin = model.rotary_emb(pad_time)
                _, attn_w = layer.self_attn(
                    enc_output, enc_output, enc_output,
                    attn_mask, cos=cos, sin=sin, output_weight=True)
            else:
                norm_times = model._normalize_timestamps(pad_time)
                _, attn_w = layer.self_attn(
                    enc_output, enc_output, enc_output, attn_mask,
                    time_seqs=norm_times,
                    thetas=model.hope_emb.thetas,
                    theta_prime=model.hope_emb.theta_prime,
                    output_weight=True)

            attn_w    = attn_w.mean(dim=1).cpu()
            pad_time_cpu = pad_time.cpu()
            mask_cpu  = mask.cpu()
            B, L = pad_time_cpu.shape
            for b in range(B):
                if sample_count >= max_samples:
                    break
                seq_len = int(mask_cpu[b].sum().item())
                for i in range(seq_len):
                    for j in range(i):
                        lag = float(pad_time_cpu[b, i] - pad_time_cpu[b, j])
                        w   = float(attn_w[b, i, j])
                        if lag > 0:
                            lags.append(lag)
                            weights.append(w)
                sample_count += 1
    return np.array(lags), np.array(weights)


def bin_attention(lags, weights, n_bins=40):
    max_lag = np.quantile(lags, 0.98)
    edges = np.linspace(0, max_lag, n_bins + 1)
    centers = (edges[:-1] + edges[1:]) / 2
    means, stds = [], []
    for i in range(n_bins):
        m = (lags >= edges[i]) & (lags < edges[i+1])
        means.append(weights[m].mean() if m.sum() > 5 else np.nan)
        stds.append(weights[m].std() if m.sum() > 5 else np.nan)
    return centers, np.array(means), np.array(stds)


print('Extracting attention weights...')
r_lags, r_weights = extract_attention_vs_lag(rothp, test_extrap_loader, model_type='rothp')
h_lags, h_weights = extract_attention_vs_lag(hothp, test_extrap_loader, model_type='hothp')

max_lag_plot = max(np.quantile(r_lags, 0.98), np.quantile(h_lags, 0.98))
r_c, r_m, r_s = bin_attention(r_lags, r_weights)
h_c, h_m, h_s = bin_attention(h_lags, h_weights)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(r_c, r_m, color='#4C72B0', lw=1.8, label='RoTHP')
axes[0].fill_between(r_c, r_m - r_s, r_m + r_s, color='#4C72B0', alpha=0.15)
axes[0].plot(h_c, h_m, color='#C44E52', lw=1.8, label='HoTHP')
axes[0].fill_between(h_c, h_m - h_s, h_m + h_s, color='#C44E52', alpha=0.15)
axes[0].axvline(TRAIN_MAX_EVENTS - 1, color='green', ls='--', alpha=0.7,
                label=f'Training max lag ({TRAIN_MAX_EVENTS - 1})')
axes[0].set_xlabel('Temporal Lag (normalized)')
axes[0].set_ylabel('Mean Attention Weight')
axes[0].set_title('Learned attention vs lag\n(green = training boundary)')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Zoom: OOD zone only (lag > training max)
ood_mask_r = r_lags > (TRAIN_MAX_EVENTS - 1)
ood_mask_h = h_lags > (TRAIN_MAX_EVENTS - 1)
if ood_mask_r.sum() > 100 and ood_mask_h.sum() > 100:
    r_c2, r_m2, r_s2 = bin_attention(r_lags[ood_mask_r], r_weights[ood_mask_r])
    h_c2, h_m2, h_s2 = bin_attention(h_lags[ood_mask_h], h_weights[ood_mask_h])
    axes[1].plot(r_c2, r_m2, color='#4C72B0', lw=1.8, label='RoTHP')
    axes[1].fill_between(r_c2, r_m2 - r_s2, r_m2 + r_s2, color='#4C72B0', alpha=0.15)
    axes[1].plot(h_c2, h_m2, color='#C44E52', lw=1.8, label='HoTHP')
    axes[1].fill_between(h_c2, h_m2 - h_s2, h_m2 + h_s2, color='#C44E52', alpha=0.15)
    axes[1].set_xlabel('Temporal Lag (normalized)')
    axes[1].set_ylabel('Mean Attention Weight')
    axes[1].set_title(f'OOD zone only (lag > {TRAIN_MAX_EVENTS - 1})')
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3)
else:
    axes[1].text(0.5, 0.5, 'Insufficient OOD lag pairs', ha='center', va='center',
                 transform=axes[1].transAxes)

plt.suptitle('Learned Attention Profile: RoTHP vs HoTHP on long sequences', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('Attention_Profile_Length_Extrap.png', dpi=300, bbox_inches='tight')
plt.show()